Mount Google Drive

In [7]:
import os
import re
import glob
import shutil
import unicodedata

import pandas as pd
import pyarrow.parquet as pq

from huggingface_hub import hf_hub_download
from google.colab import drive

# Flush and unmount any existing mount connections to clear the directory
try:
    drive.flush_and_unmount()
except Exception:
    pass

# Safely remove the mountpoint folder if it exists as an empty directory or leftover
try:
    if os.path.exists('/content/drive'):
        shutil.rmtree('/content/drive')
except Exception:
    pass

# Mount Google Drive freshly
drive.mount('/content/drive', force_remount=True)

ROOT = "/content/drive/MyDrive/fever_project"

os.makedirs(f"{ROOT}/data/evidence", exist_ok=True)
os.makedirs(f"{ROOT}/data/processed", exist_ok=True)

print("Project directory:", ROOT)
print("Evidence directory ready.")

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
Project directory: /content/drive/MyDrive/fever_project
Evidence directory ready.


Load training and validation datasets

In [8]:
train_path = (
    "hf://datasets/fever/fever@refs/convert/parquet/"
    "v1.0/train/0000.parquet"
)

val_path = (
    "hf://datasets/fever/fever@refs/convert/parquet/"
    "v1.0/paper_dev/0000.parquet"
)

train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)

print("Training:", train_df.shape)
print("Validation:", val_df.shape)

Training: (311431, 7)
Validation: (18999, 7)


Standardize wikipedia titles

In [9]:
BR = {
    "-LRB-": "(",
    "-RRB-": ")",
    "-LSB-": "[",
    "-RSB-": "]",
    "-LCB-": "{",
    "-RCB-": "}"
}

def nfc(s):
    return unicodedata.normalize("NFC", s)

def clean(t):
    for k, v in BR.items():
        t = t.replace(k, v)

    return re.sub(r"\s+", " ", t).strip()

Identify the Wikipedia pages we need

In [10]:
def needed_pages(df):
    d = df[
        (df["label"] != "NOT ENOUGH INFO") &
        (df["evidence_wiki_url"] != "")
    ]

    return set(d["evidence_wiki_url"].map(nfc))


need_train = needed_pages(train_df)
need_val = needed_pages(val_df)

need_all = need_train | need_val

print("TRAIN REQUIRED PAGES:", len(need_train))
print("VALIDATION REQUIRED PAGES:", len(need_val))
print("OVERLAPPING PAGES:", len(need_train & need_val))
print("TOTAL UNIQUE REQUIRED PAGES:", len(need_all))

TRAIN REQUIRED PAGES: 12549
VALIDATION REQUIRED PAGES: 1460
OVERLAPPING PAGES: 451
TOTAL UNIQUE REQUIRED PAGES: 13558


Extract Wikipedia sentences

In [11]:
for k in range(10):

    out = f"{ROOT}/data/evidence/part_{k}.parquet"

    # Skip files that have already been processed
    if os.path.exists(out):
        print("Already exists, skipping:", k)
        continue

    print(f"\nProcessing Wikipedia file {k:04d}...")

    local = hf_hub_download(
        repo_id="fever/fever",
        filename=f"wiki_pages/partial-wikipedia_pages/{k:04d}.parquet",
        repo_type="dataset",
        revision="refs/convert/parquet",
        cache_dir="/content/hf_tmp"
    )

    rows = []

    parquet_file = pq.ParquetFile(local)

    for batch in parquet_file.iter_batches(
        batch_size=20000,
        columns=["id", "lines"]
    ):

        page_ids = batch.column("id").to_pylist()
        page_lines = batch.column("lines").to_pylist()

        for pid, lines in zip(page_ids, page_lines):

            pid = nfc(pid)

            if pid not in need_all:
                continue

            for line in lines.split("\n"):

                parts = line.split("\t")

                if (
                    len(parts) >= 2
                    and parts[0].isdigit()
                    and parts[1].strip()
                ):

                    rows.append((
                        pid,
                        int(parts[0]),
                        clean(parts[1])
                    ))

    part_df = pd.DataFrame(
        rows,
        columns=["page", "sent_id", "text"]
    )

    part_df.to_parquet(out, index=False)

    print("Completed part:", k)
    print("Sentences extracted:", len(part_df))

    # Remove temporary download cache to free Colab storage
    shutil.rmtree("/content/hf_tmp", ignore_errors=True)

Already exists, skipping: 0
Already exists, skipping: 1
Already exists, skipping: 2
Already exists, skipping: 3
Already exists, skipping: 4
Already exists, skipping: 5
Already exists, skipping: 6
Already exists, skipping: 7
Already exists, skipping: 8
Already exists, skipping: 9


Combine the extracted evidence

In [26]:
sent = pd.concat(
    [
        pd.read_parquet(f)
        for f in sorted(
            glob.glob(f"{ROOT}/data/evidence/part_*.parquet")
        )
    ],
    ignore_index=True
)

# Remove duplicate page and sentence combinations
sent = sent.drop_duplicates(["page", "sent_id"])

print("TOTAL EXTRACTED SENTENCES:", len(sent))
print("UNIQUE WIKIPEDIA PAGES:", sent["page"].nunique())

display(sent.head())

TOTAL EXTRACTED SENTENCES: 130699
UNIQUE WIKIPEDIA PAGES: 15315


,page,sent_id,text
0,19_Kids_and_Counting,0,19 Kids and Counting ( formerly 17 Kids and Co...
1,19_Kids_and_Counting,1,The show featured the Duggar family : parents ...
2,19_Kids_and_Counting,4,The reality show focused on the life of the Du...
3,19_Kids_and_Counting,5,"and frequently discussed values of purity , mo..."
4,19_Kids_and_Counting,6,The Duggars avoid birth control saying they ha...


In [17]:
import json, zipfile, requests

out = f"{ROOT}/data/evidence/part_10.parquet"
missing = need_all - set(sent["page"])
print(f"{len(missing):,} required pages still missing")

if os.path.exists(out):
    print("part_10 already exists, skipping")
else:
    zip_path = "/content/wiki-pages.zip"
    if not os.path.exists(zip_path):
        with requests.get("https://fever.ai/download/fever/wiki-pages.zip",
                          stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(zip_path, "wb") as f:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk)
    print("Zip ready")

    rows, found, bad_lines = [], set(), 0
    with zipfile.ZipFile(zip_path) as zf:
        names = sorted(n for n in zf.namelist() if n.endswith(".jsonl"))
        print(f"{len(names)} jsonl files in zip")
        for n in names:
            with zf.open(n) as fh:
                for line in fh:
                    line = line.strip()
                    if not line:
                        continue                      # skip blank lines
                    try:
                        d = json.loads(line)
                    except json.JSONDecodeError:
                        bad_lines += 1                # skip any corrupt line
                        continue
                    pid = nfc(d["id"])
                    if pid not in missing:
                        continue
                    found.add(pid)
                    for ln in d["lines"].split("\n"):
                        parts = ln.split("\t")
                        if len(parts) >= 2 and parts[0].isdigit() and parts[1].strip():
                            rows.append((pid, int(parts[0]), clean(parts[1])))
            print(n, f"{len(found):,}/{len(missing):,} pages found", end="\r")

    part_df = pd.DataFrame(rows, columns=["page", "sent_id", "text"])
    part_df.to_parquet(out, index=False)
    os.remove(zip_path)
    print(f"\nbad lines skipped: {bad_lines}")
    print(f"part_10.parquet: {len(part_df):,} sentences from {len(found):,} pages")

6,293 required pages still missing
Zip ready
218 jsonl files in zip

bad lines skipped: 109
part_10.parquet: 51,869 sentences from 6,293 pages


In [24]:
import requests, json
r = requests.get("https://fever.ai/download/fever/train.jsonl", timeout=300)
r.raise_for_status()
raw = {}
for line in r.text.splitlines():
    if line.strip():
        d = json.loads(line)
        raw[d["id"]] = d

print(json.dumps(raw[7], indent=1)[:800])

def no_sentence(d):
    if d["label"] == "NOT ENOUGH INFO":
        return False
    return all(item[3] is None for s in d["evidence"] for item in s)

n = sum(no_sentence(d) for d in raw.values())
print("Claims in original train.jsonl with no sentence id:", n, "(Hugging Face table: 32,219)")

{
 "id": 7,
 "verifiable": "VERIFIABLE",
 "label": "SUPPORTS",
 "claim": "Roald Dahl is a writer.",
 "evidence": [
  [
   [
    15234,
    18987,
    "Roald_Dahl",
    0
   ]
  ]
 ]
}
Claims in original train.jsonl with no sentence id: 0 (Hugging Face table: 32,219)


In [25]:
# Original FEVER rows -> same columns as the Hugging Face tables
def flatten(claims):
    rows = []
    for d in claims:
        for ev_set in d["evidence"]:
            for ann, ev, page, sid in ev_set:
                rows.append((
                    d["id"], d["label"], d["claim"],
                    -1 if ann is None else ann,
                    -1 if ev is None else ev,
                    page or "",
                    -1 if sid is None else sid,
                ))
    return pd.DataFrame(rows, columns=[
        "id", "label", "claim", "evidence_annotation_id",
        "evidence_id", "evidence_wiki_url", "evidence_sentence_id"])

r = requests.get("https://fever.ai/download/fever/paper_dev.jsonl", timeout=300)
r.raise_for_status()
val_raw = [json.loads(l) for l in r.text.splitlines() if l.strip()]

train_df_hf, val_df_hf = train_df, val_df          # keep the old tables as backup
train_df = flatten(raw.values())
val_df   = flatten(val_raw)

for name, d in (("train", train_df), ("val", val_df)):
    print(f"{name}: rows={len(d):,} unique claims={d['id'].nunique():,}")
    print(d["label"].value_counts(normalize=True).round(4).to_dict())
    sr = d[d["label"] != "NOT ENOUGH INFO"]
    print("  SUPPORTS/REFUTES rows with sentence_id < 0:", (sr["evidence_sentence_id"] < 0).sum())

# Recompute required pages from the correct data
need_train = needed_pages(train_df)
need_val   = needed_pages(val_df)
need_all   = need_train | need_val
print("\nTRAIN REQUIRED PAGES:", len(need_train))
print("VALIDATION REQUIRED PAGES:", len(need_val))
print("TOTAL UNIQUE REQUIRED PAGES:", len(need_all))

still_missing = need_all - set(sent["page"])
print("Required pages not yet extracted:", len(still_missing))

# Save the corrected claim tables so Stage 4 and the other members use them
train_df.to_parquet(f"{ROOT}/data/processed/claims_train_flat.parquet", index=False)
val_df.to_parquet(f"{ROOT}/data/processed/claims_val_flat.parquet", index=False)

train: rows=311,431 unique claims=145,449
{'SUPPORTS': 0.6221, 'REFUTES': 0.225, 'NOT ENOUGH INFO': 0.1529}
  SUPPORTS/REFUTES rows with sentence_id < 0: 0
val: rows=18,999 unique claims=9,999
{'SUPPORTS': 0.3865, 'REFUTES': 0.3754, 'NOT ENOUGH INFO': 0.2381}
  SUPPORTS/REFUTES rows with sentence_id < 0: 0

TRAIN REQUIRED PAGES: 12549
VALIDATION REQUIRED PAGES: 1460
TOTAL UNIQUE REQUIRED PAGES: 13558
Required pages not yet extracted: 0


Check evidence coverage

In [27]:
found_pages = set(sent["page"])

keys = set(
    zip(sent["page"], sent["sent_id"])
)

In [28]:
def coverage(df, need, name):

    d = df[
        df["label"] != "NOT ENOUGH INFO"
    ].copy()

    d["page"] = d["evidence_wiki_url"].map(nfc)

    d["ok"] = [
        (p, s) in keys
        for p, s in zip(
            d["page"],
            d["evidence_sentence_id"]
        )
    ]

    # A claim is covered only when all its evidence sentences exist
    claim_ok = d.groupby("id")["ok"].all()

    label_of = (
        d.drop_duplicates("id")
        .set_index("id")["label"]
    )

    print(f"\n[{name}]")
    print(
        "Pages found:",
        f"{len(need & found_pages) / len(need):.3f}"
    )

    print(
        "Evidence sentences found:",
        f"{d['ok'].mean():.3f}"
    )

    print(
        "Claims with ALL evidence found:",
        f"{claim_ok.mean():.3f}",
        f"({claim_ok.sum()} of {len(claim_ok)})"
    )

    print("\nCoverage by label:")
    print(claim_ok.groupby(label_of).mean())

    print("\nMost frequently missing pages:")

    return d[~d["ok"]]["page"].value_counts().head(10)

In [29]:
missing_train = coverage(
    train_df,
    need_train,
    "TRAIN"
)

missing_val = coverage(
    val_df,
    need_val,
    "VALIDATION"
)

print("\nMISSING TRAIN PAGES:")
print(missing_train)

print("\nMISSING VALIDATION PAGES:")
print(missing_val)


[TRAIN]
Pages found: 1.000
Evidence sentences found: 1.000
Claims with ALL evidence found: 1.000 (109810 of 109810)

Coverage by label:
label
REFUTES     1.0
SUPPORTS    1.0
Name: ok, dtype: float64

Most frequently missing pages:

[VALIDATION]
Pages found: 1.000
Evidence sentences found: 1.000
Claims with ALL evidence found: 1.000 (6666 of 6666)

Coverage by label:
label
REFUTES     1.0
SUPPORTS    1.0
Name: ok, dtype: float64

Most frequently missing pages:

MISSING TRAIN PAGES:
Series([], Name: count, dtype: int64)

MISSING VALIDATION PAGES:
Series([], Name: count, dtype: int64)


In [30]:
import glob

RES = f"{ROOT}/results"
os.makedirs(RES, exist_ok=True)

def valid_rows(df):
    d = df[(df["label"] != "NOT ENOUGH INFO") &
           (df["evidence_wiki_url"] != "") &
           (df["evidence_sentence_id"] >= 0)].copy()
    d["page"] = d["evidence_wiki_url"].map(nfc)
    d["sent_id"] = d["evidence_sentence_id"]
    return d

stats, label_rows = {}, []
for name, df_ in (("train", train_df), ("val", val_df)):
    d = valid_rows(df_)

    # sentences_{split}.parquet = only the gold evidence sentences for this split
    ptr = d[["page", "sent_id"]].drop_duplicates()
    out_df = ptr.merge(sent, on=["page", "sent_id"], how="inner") \
                .sort_values(["page", "sent_id"]).reset_index(drop=True)
    out_df.to_parquet(f"{ROOT}/data/evidence/sentences_{name}.parquet", index=False)

    d["ok"] = [(p, s) in keys for p, s in zip(d["page"], d["sent_id"])]
    claim_ok = d.groupby("id")["ok"].all()
    claim_label = df_.drop_duplicates("id").set_index("id")["label"]
    need = need_train if name == "train" else need_val

    stats[name] = dict(
        claims=df_["id"].nunique(),
        evid_claims=len(claim_ok),
        pages=len(need), pages_found=len(need & found_pages),
        pointers=len(ptr), pointers_found=len(out_df),
        claims_all=int(claim_ok.sum()),
        nei=int((claim_label == "NOT ENOUGH INFO").sum()),
    )
    for lab, n in claim_label.value_counts().items():
        got = claim_ok[claim_ok.index.map(claim_label) == lab]
        label_rows.append((name, lab, int(n), len(got), int(got.sum())))
    print(f"sentences_{name}.parquet: {len(out_df):,} rows")

s_tr, s_va = stats["train"], stats["val"]
overlap = len(need_train & need_val)
n_parts = len(glob.glob(f"{ROOT}/data/evidence/part_*.parquet"))

lab_tbl = "\n".join(f"| {s} | {l} | {c:,} | {e:,} | {a:,} |" for s, l, c, e, a in sorted(label_rows))

report = f"""# Wikipedia Evidence Coverage Report (Member 1)

Validation split: paper_dev | Evidence part files: {n_parts} (part_0-9 from the Hugging Face partial parquet conversion, plus part_10 from the original FEVER wiki-pages.zip)

## Required pages
- Train: {s_tr['pages']:,} pages; Validation: {s_va['pages']:,} pages
- Union extracted: {len(need_all):,} pages
- Overlap: {overlap:,} pages ({overlap/s_va['pages']:.1%} of validation pages are also needed by training)

## Coverage
| Metric | Train | Validation |
|---|---|---|
| Required pages found | {s_tr['pages_found']:,} / {s_tr['pages']:,} ({s_tr['pages_found']/s_tr['pages']:.1%}) | {s_va['pages_found']:,} / {s_va['pages']:,} ({s_va['pages_found']/s_va['pages']:.1%}) |
| Evidence sentences found | {s_tr['pointers_found']:,} / {s_tr['pointers']:,} ({s_tr['pointers_found']/s_tr['pointers']:.1%}) | {s_va['pointers_found']:,} / {s_va['pointers']:,} ({s_va['pointers_found']/s_va['pointers']:.1%}) |
| SUPPORTS/REFUTES claims with ALL evidence found | {s_tr['claims_all']:,} / {s_tr['evid_claims']:,} ({s_tr['claims_all']/s_tr['evid_claims']:.1%}) | {s_va['claims_all']:,} / {s_va['evid_claims']:,} ({s_va['claims_all']/s_va['evid_claims']:.1%}) |
| NOT ENOUGH INFO claims (no evidence needed) | {s_tr['nei']:,} | {s_va['nei']:,} |

## Coverage by label (claims)
| Split | Label | Claims | Claims needing evidence | Claims with all evidence found |
|---|---|---|---|---|
{lab_tbl}

## Important data note
The Hugging Face `fever/fever` v1.0 claim tables have `evidence_sentence_id = -1` for about 32,000 train and 2,500 validation SUPPORTS/REFUTES claims. The original FEVER `train.jsonl` has real sentence ids for those claims, so all claim tables were rebuilt from the original jsonl files (saved as `data/processed/claims_train_flat.parquet` and `claims_val_flat.parquet`). Use these files, not the Hugging Face tables, for gold pairs.

## Notes for Members 2 and 3
- `part_N.parquet` holds all sentences of the required pages (`page`, `sent_id`, `text`), usable for TF-IDF retrieval.
- `sentences_train/val.parquet` hold only the gold evidence sentences.
- Sentence text is already cleaned (`-LRB-` style tokens replaced by real brackets); page titles keep the FEVER token form.
"""
open(f"{RES}/evidence_coverage_report.md", "w").write(report)
print(report)

sentences_train.parquet: 34,796 rows
sentences_val.parquet: 3,493 rows
# Wikipedia Evidence Coverage Report (Member 1)

Validation split: paper_dev | Evidence part files: 11 (part_0-9 from the Hugging Face partial parquet conversion, plus part_10 from the original FEVER wiki-pages.zip)

## Required pages
- Train: 12,549 pages; Validation: 1,460 pages
- Union extracted: 13,558 pages
- Overlap: 451 pages (30.9% of validation pages are also needed by training)

## Coverage
| Metric | Train | Validation |
|---|---|---|
| Required pages found | 12,549 / 12,549 (100.0%) | 1,460 / 1,460 (100.0%) |
| Evidence sentences found | 34,796 / 34,796 (100.0%) | 3,493 / 3,493 (100.0%) |
| SUPPORTS/REFUTES claims with ALL evidence found | 109,810 / 109,810 (100.0%) | 6,666 / 6,666 (100.0%) |
| NOT ENOUGH INFO claims (no evidence needed) | 35,639 | 3,333 |

## Coverage by label (claims)
| Split | Label | Claims | Claims needing evidence | Claims with all evidence found |
|---|---|---|---|---|
| train | N